# 10. Predict, check and explain

**Open:** restart R; save the supplied script as `scripts/my-10-test-tree.R`.

**Input:** Chapter 9's saved tree. **Today:** read the learned rule, check its predictions, compare it with the baseline and reuse it on new inputs.

## 10.1 Read the tree before running predictions

Run `prediction-package`, then the supplied `load-tree-fit` block to restore the selected tree and its training preparation:

In [1]:
library(rpart)

In [2]:
source("scripts/tree_helpers.R")
if (!file.exists("outputs/checkpoints/tree-fit.rds")) stop("Run Chapter 9 first to create tree-fit.rds.")
chosen <- readRDS("outputs/checkpoints/tree-fit.rds")
final_tree <- chosen$fit
train_filled <- fill_predictors(chosen$train, chosen$medians)

Run the supplied `plot-tree` block. Open `outputs/tree/fitted-tree.png` in Files; the code saves the image directly.

In [3]:
nodes <- extract_tree_nodes(final_tree)
write_tree_mermaid(nodes, "outputs/tree/fitted-tree.mmd")
png("outputs/tree/fitted-tree.png", width = 1200, height = 720, res = 150)
par(mar = c(0, 0, 3, 0), xpd = NA, family = "sans")
draw_teaching_tree(final_tree, nodes)
invisible(dev.off())
cat("Saved outputs/tree/fitted-tree.png\n")

Saved outputs/tree/fitted-tree.png


![Selected shallow regression tree. Each final group predicts its own mean training log10 MSC. Counts are training records, not independent farms.](../outputs/tree/fitted-tree.png)

Validation selected depth 1 here, so the final diagram matches Chapter 8's first example. The difference is that we have now checked that depth against two deeper candidates inside training farms.

**Point before you run:**

| Input change | What do you expect from this tree? |
|---|---|
| 50 cows versus 30 cows | Which branch and leaf for each? |
| 50 cows becomes 60 | Does it cross the cut? |
| Only pasture hours changes | Is that variable used anywhere in this tree? |
| 36 cows becomes 37 | Does it cross the cut? |

```{admonition} Open after tracing the branches
:class: tip dropdown
50 and 60 cows both satisfy “at least 36.5”, enter leaf 2 and receive about 0.6873 log10 MSC. 30 and 36 enter leaf 3 and receive about 1.0545; 37 enters leaf 2. Changing pasture hours alone does not change this selected tree's prediction. All records in a leaf receive the same value.
```

The cut separates 36 or fewer from 37 or more cows; it is not a half cow. Pasture hours were offered, but not selected. These are associations learned from the training records, not intervention advice or a safety threshold.

## 10.2 Follow one real record and check its leaf

**Supplied preparation — `prepare-test`:** run this block to give missing test inputs the saved **training** medians. The medians do not change after seeing test inputs.

In [4]:
# Test inputs use saved training medians, with no refit or new choice.
test_filled <- fill_predictors(chosen$test, chosen$medians)

Run `predict-tree`:

In [5]:
tree_prediction <- predict(final_tree, newdata = test_filled)
head(data.frame(observed_log10 = test_filled$log10_msc,
                tree_prediction = as.numeric(tree_prediction)), 3)

,observed_log10,tree_prediction
,<dbl>,<dbl>
1,0.8129134,0.6873386
2,0.5440680,0.6873386
3,-0.3010300,0.6873386


As in Chapter 8, `predict()` follows an existing rule; it does not fit a new one. Although this table contains known lab answers for evaluation, the fitting formula uses only pasture hours and cow count as inputs.

Run the supplied `trace-one` block. First compare recorded inputs with the model's inputs; then read the rule, leaf and prediction:

In [6]:
first_route <- trace_tree(test_filled[1, ], nodes)
data.frame(feature = tree_features, recorded = unlist(chosen$test[1, tree_features], use.names = FALSE),
           used_by_model = unlist(test_filled[1, tree_features], use.names = FALSE))
data.frame(rule = paste(first_route$path, collapse = "; "), leaf = first_route$leaf,
           prediction_log10 = first_route$prediction)
stopifnot(isTRUE(all.equal(first_route$prediction, as.numeric(tree_prediction[1]))))

feature,recorded,used_by_model
<chr>,<dbl>,<dbl>
pasture_hours,12,12
milking_cows,50,50


rule,leaf,prediction_log10
<chr>,<int>,<dbl>
milking_cows >= 36.5,2,0.6873386


The first record has **50 cows**. On the diagram, follow “Yes” to leaf 2: it predicts about **0.6873 log10 MSC**. The observed answer is about 0.8129, so observed minus predicted is about **0.1256**. Reconstruct that subtraction before looking at an overall score.

Where does 0.6873 come from? Run the supplied `check-leaves` block:

In [7]:
train_routes <- lapply(seq_len(nrow(train_filled)), function(i) trace_tree(train_filled[i, ], nodes))
train_leaf <- vapply(train_routes, function(route) route$leaf, integer(1))
train_prediction <- as.numeric(predict(final_tree, newdata = train_filled))
stopifnot(isTRUE(all.equal(vapply(train_routes, function(route) route$prediction, numeric(1)), train_prediction)))
leaf_means <- aggregate(train_filled$log10_msc, list(leaf = train_leaf), mean)
names(leaf_means)[2] <- "mean_from_training_records"
leaf_means$tree_prediction <- nodes$mean_log10[match(leaf_means$leaf, nodes$node)]
stopifnot(isTRUE(all.equal(leaf_means$mean_from_training_records, leaf_means$tree_prediction)))
leaf_means

leaf,mean_from_training_records,tree_prediction
<int>,<dbl>,<dbl>
2,0.6873386,0.6873386
3,1.0545454,1.0545454


One column calculates each leaf's average from its training records; the other reads the model's prediction. They agree. The 354 training records in leaf 2 supplied its average. The new test record follows the rule but does not join the average or change it.

## 10.3 Compare predictions on the same test records

**Before running:** what evidence would justify the more complicated rule?

Run `tree-errors`. The three lines calculate residuals, RMSE and MAE:

In [8]:
tree_errors <- test_filled$log10_msc - tree_prediction
sqrt(mean(tree_errors^2))
mean(abs(tree_errors))

[1] 0.544897

[1] 0.4264264

Then run the supplied `tree-test-scores` block to place those scores beside the Chapter 7 baseline:

In [9]:
baseline_prediction <- rep(mean(chosen$train$log10_msc), nrow(test_filled))
tree_scores <- data.frame(model = c("Training-mean baseline", "Shallow regression tree"),
  maxdepth = c(0, chosen$chosen_depth), n_test = nrow(test_filled), test_farms = length(unique(test_filled$FarmID)),
  RMSE_log10_CFU_ml = c(sqrt(mean((test_filled$log10_msc - baseline_prediction)^2)), sqrt(mean(tree_errors^2))),
  MAE_log10_CFU_ml = c(mean(abs(test_filled$log10_msc - baseline_prediction)), mean(abs(tree_errors))))
tree_scores[c("model", "RMSE_log10_CFU_ml", "MAE_log10_CFU_ml")]

model,RMSE_log10_CFU_ml,MAE_log10_CFU_ml
<chr>,<dbl>,<dbl>
Training-mean baseline,0.5289231,0.4091480
Shallow regression tree,0.5448970,0.4264264


Both methods predict the **same 105 records from the same 21 held-out farms**, on the same log10 scale. Lower errors are better. Tree RMSE is about **0.545**, compared with **0.529** for the baseline; its MAE is also higher. This tree did not improve these test predictions. The baseline was also slightly better in the training-only OOF comparison.

That is an explainable result. Two survey variables and a shallow tree did not earn extra complexity here. It does not show that all trees fail or that the inputs lack biological relevance. Keep the result rather than repeatedly changing the analysis to improve a visible test score. The test split was already displayed in the baseline lesson, so this is a fixed teaching comparison, not a fresh blinded evaluation.

**Supplied evidence export — `save-tree-results`:** run this block to save the comparisons and prediction checks.

In [10]:
write.csv(tree_scores, "outputs/tree/test-scores.csv", row.names = FALSE)
write.csv(nodes, "outputs/tree/nodes.csv", row.names = FALSE)
write.csv(leaf_means, "outputs/tree/leaf-means.csv", row.names = FALSE)
write.csv(data.frame(feature = tree_features, median = unname(chosen$medians)),
          "outputs/tree/training-medians.csv", row.names = FALSE)
training_predictions <- data.frame(FarmID = train_filled$FarmID, Sampling = train_filled$Sampling,
  observed_log10 = train_filled$log10_msc, train_filled[tree_features], leaf = train_leaf,
  tree_prediction = train_prediction)
write.csv(training_predictions, "outputs/tree/training-predictions.csv", row.names = FALSE)
test_predictions <- data.frame(FarmID = test_filled$FarmID, Sampling = test_filled$Sampling,
  observed_log10 = test_filled$log10_msc, test_filled[tree_features],
  pasture_hours_missing = is.na(chosen$test$pasture_hours), milking_cows_missing = is.na(chosen$test$milking_cows),
  tree_prediction = as.numeric(tree_prediction), baseline_prediction = baseline_prediction)
write.csv(test_predictions, "outputs/tree/test-predictions.csv", row.names = FALSE)
writeLines(capture.output(sessionInfo()), "outputs/tree/session-info.txt")

## 10.4 Save, reload and predict new inputs

A reusable prediction needs both the learned tree and the references for preparing missing inputs. A **list** bundles named R objects together; `$fit` later retrieves the item named `fit`.

Run `save-model`. `saveRDS()` writes the bundle to a file:

In [11]:
model_for_reuse <- list(fit = final_tree, medians = chosen$medians, features = tree_features)
saveRDS(model_for_reuse, "outputs/tree/model.rds")

Run `reload-model`. `readRDS()` reads it back into R:

In [12]:
saved_model <- readRDS("outputs/tree/model.rds")

These operations save and restore a rule, not train another tree.

Run `new-samples` to create the two **invented** input profiles from Chapter 8. Neither has an observed MSC:

In [13]:
# Invented input profiles, without observed outcomes; not an evaluation dataset.
new_samples <- data.frame(pasture_hours = c(6, 6), milking_cows = c(50, 30))
new_samples

pasture_hours,milking_cows
<dbl>,<dbl>
6,50
6,30


**Predict first:** at 6 pasture hours, which leaf will 50 cows enter? Which will 30 cows enter?

Run the supplied `prepare-new` block. It applies the saved missing-input rule; neither of these complete rows needs filling.

In [14]:
new_ready <- fill_predictors(new_samples, saved_model$medians)

Run `new-predict`:

In [15]:
new_prediction <- predict(saved_model$fit, newdata = new_ready)
example_predictions <- data.frame(new_samples, prediction_log10 = as.numeric(new_prediction),
                                  concentration_expression_CFU_ml = 10^as.numeric(new_prediction))
example_predictions
write.csv(example_predictions, "outputs/tree/example-predictions.csv", row.names = FALSE)

pasture_hours,milking_cows,prediction_log10,concentration_expression_CFU_ml
<dbl>,<dbl>,<dbl>,<dbl>
6,50,0.6873386,4.867866
6,30,1.0545454,11.338235


The predictions are about 0.6873 and 1.0545 log10 MSC, just as the diagram suggests. The last column expresses them as `10^prediction` CFU/mL, as in Chapter 7. It is not an arithmetic-mean concentration or a safety verdict. With no observed outcomes, these two rows provide no accuracy score.

**Change one thing in your my- copy:** in `new-samples`, change 50 cows to 60. Predict whether the result changes. Then rerun **new-samples → prepare-new → new-predict**. Changing the input alone does not automatically refresh an old prediction. Next try 30 cows to 37, or change only pasture hours. Explain each result using the branch, rather than asking AI to invent a biological story. Comparing invented inputs is not an intervention experiment.

## 10.5 Ask AI for two familiar operations

Use the course-approved chat and your **my- copy**, keeping the supplied reference script intact. The chat cannot inspect your R session unless you provide its contents.

Give AI this small, explicit task:

> I am using R and rpart. My prepared training data frame is train_filled. The numeric target is log10_msc; the only predictors are pasture_hours and milking_cows. selected_settings already contains my training-validation choice. My prepared held-out data frame is test_filled. Write only the code to fit a numeric regression tree called ai_tree on train_filled, then predict test_filled into ai_prediction. Do not change the settings, split or preparation. Do not use held-out outcomes for fitting or choosing settings. Explain which line learns and which line only predicts.

**Inspect before running:**

| Check in the returned code | Why it matters |
|---|---|
| Fitting `data` is `train_filled` | Hidden answers must not train the model |
| Left of `~` is `log10_msc`; right lists only the two inputs | The answer and farm labels must not become predictors |
| Numeric method and existing `selected_settings` are used | This is the procedure already chosen in training |
| `predict()` uses `newdata = test_filled` | We need predictions for those new rows |

Run `ai-settings` to recreate the settings name in this session:

In [16]:
selected_settings <- fixed_tree_settings
selected_settings$maxdepth <- chosen$chosen_depth
# My live AI attempt: add returned code and your check below in your my- copy.

In your my- script, find the **My live AI attempt** comment at the end. Add the AI code below it, then add this check comparing its predictions with the reference `tree_prediction`. Run your added lines after completing the earlier blocks:

```r
all.equal(as.numeric(ai_prediction), as.numeric(tree_prediction))
```

For an equivalent fit, expect TRUE. If it differs or errors, give AI the actual code and message and investigate; do not change the test answers or invent a successful check. Append the prompt, returned code, your check and any repair to your AI-use log; keep your Chapter 3 record. If you cannot access a chat, inspect the visible `fit-tree` and `predict-tree` blocks and label this as a code-inspection exercise rather than a completed AI interaction.

## 10.6 Keep the evidence and transfer the workflow

| Find in outputs/tree | What it lets you explain |
|---|---|
| `predictor-audit.csv` | Missing survey information and records kept |
| `validation-scores.csv` | Why this depth was selected |
| `test-scores.csv` | Same-record comparison with the baseline |
| `leaf-means.csv` and `nodes.csv` | Where the predictions came from |
| `model.rds` and `example-predictions.csv` | Reuse of the rule and its preparation |

**Write three sentences:** name the inputs and validation choice; compare the two test errors; state one limit. Missing survey answers were filled by a simple assumption, samples repeat within farms, and this split cannot establish performance in another region, causal effects or milk safety.

| Step | When the algorithm changes |
|---|---|
| Define the task | Keep the target, available information and intended new-farm setting explicit |
| Prepare inputs | A new method may need different preparation; learn it inside each fitting partition |
| Fit and choose settings | Replace the fitting function and settings; keep whole-farm training validation |
| Predict and compare | Predict appropriate held-out records, without using their answers to choose the procedure |
| Explain and verify | Keep predictions and checks; a different model needs an explanation suited to its own rules |

**Finish:** save optional changed-input results under another filename if you want to keep them. Restart R and run a my- copy of `scripts/full-analysis.R`; it recreates the supplied reference baseline and tree from the original downloads. Explain which steps you can reuse when requesting another model from AI.